# Adoption Economics · 2D Wizard Game · Photonic AUV · Fly Me to the Moon

> Late adopter of great hardware — economically honest. Wizard game engine.
> 3D photonic AUV. Jalali art. SE Asia space programs. Orbital mechanics.

| § | Domain | Core |
|---|---|---|
| §1 | 💰 Adoption Economics | Rogers S-curve · NPV · real options · Moore's law photonics |
| §2 | 🎮 2D Wizard Game | Projectile physics · robot AI · skin passport · dungeon gen |
| §3 | 🤿 Photonic AUV | 3D hull design · sensor payload · buoyancy/drag · Jalali art |
| §4 | 🌙 Fly Me to the Moon | Hohmann transfer · ΔV budget · lunar trajectory · Frank |
| §5 | 🌏 SE Asia Space | VNSC · PhilSA · Folsom demographics · customs flow model |


In [ ]:
import sympy as sp
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from matplotlib.colors import LightSource
from scipy.integrate import solve_ivp, odeint
from scipy.stats import norm
from scipy.optimize import brentq
from IPython.display import display, Math
import warnings, random, time
warnings.filterwarnings('ignore')
sp.init_printing(use_latex='mathjax')
plt.rcParams.update({'font.size':10,'figure.dpi':100})
np.random.seed(2026)
print('✦ all modules loaded')

---
# §1 💰 Honest Adoption Economics — "Hardware Is So Good, Adoption Is Inevitable"

**The argument:** You're not the original adopter (Type 1). Hardware costs are high now.
But if adoption is inevitable — the NPV question is *when*, not *if*.  

**Rogers diffusion:** $F(t)=\\frac{1}{1+e^{-k(t-t_0)}}$ (S-curve).  
**Cost trajectory:** hardware prices fall as $C(t)=C_0\\cdot e^{-\\lambda t}$ (Moore's law analog).  
**NPV of adopting at time $\\tau$:**
$$\\text{NPV}(\\tau) = \\int_\\tau^T \\frac{B(t)-C(t)}{(1+r)^{t-\\tau}}\\,dt - C_{\\text{purchase}}(\\tau)$$

**Optimal adoption time:** $\\tau^* = \\arg\\max_\\tau\\,\\text{NPV}(\\tau)$.

In [ ]:
# ── SymPy: NPV integral and optimal adoption ──────────────────────
tau_s,t_s,T_s,r_s    = sp.symbols('tau t T r', positive=True)
C0_s,lam_s,B0_s,k_s  = sp.symbols('C_0 lambda B_0 k', positive=True)
t0_s = sp.Symbol('t_0', real=True)

# Benefit B(t): grows with market adoption S-curve
F_adopt = 1/(1+sp.exp(-k_s*(t_s-t0_s)))   # S-curve fraction
B_t     = B0_s * F_adopt
# Cost falls exponentially
C_pur   = C0_s * sp.exp(-lam_s*tau_s)

print('Technology adoption model:')
display(Math(r'F(t) = \frac{1}{1+e^{-k(t-t_0)}} \quad (\text{Rogers S-curve})'))  
display(Math(r'C_{\rm purchase}(\tau) = C_0 e^{-\lambda\tau} \quad (\text{cost decay})'))  
display(Math(r'B(t) = B_0 F(t) \quad (\text{benefit flows with adoption})'))  

# ── Numerical: NPV surface ─────────────────────────────────────────
# Photonic sensing hardware parameters (realistic 2024-2035 scenario)
C0_v   = 150_000   # $150k initial hardware cost (2024)
lam_v  = 0.12      # 12%/yr cost decay (Moore-like for photonics)
B0_v   = 80_000    # $80k/yr max benefit (when fully adopted)
k_v    = 0.8       # diffusion speed
t0_v   = 5.0       # inflection at year 5 (2029)
r_v    = 0.08      # 8% discount rate
T_v    = 12.0      # 12-year horizon

def npv(tau, C0, lam, B0, k, t0, r, T, dt=0.05):
    """NPV of adopting at time tau."""
    t_arr = np.arange(tau, T, dt)
    if len(t_arr)==0: return -C0*np.exp(-lam*tau)
    F_arr = 1/(1+np.exp(-k*(t_arr-t0)))
    B_arr = B0 * F_arr
    # Maintenance cost: 10% of purchase price/yr, declining
    maint = 0.10 * C0*np.exp(-lam*tau)
    pv    = np.sum((B_arr - maint) / (1+r)**(t_arr-tau)) * dt
    return pv - C0*np.exp(-lam*tau)

tau_arr = np.linspace(0, T_v-1, 200)
npv_arr = np.array([npv(t,C0_v,lam_v,B0_v,k_v,t0_v,r_v,T_v) for t in tau_arr])
tau_star= tau_arr[np.argmax(npv_arr)]

# Type profiles
adopter_types = {
    'Innovator (2.5%)':      dict(tau=0.5,  color='#e41a1c', marker='*',  ms=12),
    'Early Adopter (13.5%)': dict(tau=2.0,  color='#ff7f00', marker='^',  ms=10),
    'Early Majority (34%)':  dict(tau=4.5,  color='#4daf4a', marker='s',  ms=9),
    'Late Majority (34%)':   dict(tau=7.0,  color='#377eb8', marker='D',  ms=9),
    'Laggard (16%)':         dict(tau=10.0, color='#984ea3', marker='v',  ms=9),
}
# Optimal: computed above

print(f'\nAdoption Economics (C₀=${C0_v:,.0f}, λ={lam_v*100:.0f}%/yr, T={T_v:.0f}yr):')
print(f'{"Adopter type":28}  {"τ (yr)":8}  {"NPV ($)"}:  {"Decision"}')
for atype, ap in adopter_types.items():   # loop: adopter types
    npv_v = npv(ap['tau'],C0_v,lam_v,B0_v,k_v,t0_v,r_v,T_v)
    cost_v= C0_v*np.exp(-lam_v*ap['tau'])
    dec   = '✓ ADOPT' if npv_v>0 else '✗ WAIT'
    print(f'{atype:28}  {ap["tau"]:8.1f}  ${npv_v:>10,.0f}   {dec}  (HW=${cost_v:,.0f})')
print(f'\n→ OPTIMAL adoption: τ* = {tau_star:.1f} yr  (NPV = ${npv_arr.max():,.0f})')
print(f'  Hardware cost at τ*: ${C0_v*np.exp(-lam_v*tau_star):,.0f}  '
      f'(vs ${C0_v:,} today = {100*(1-np.exp(-lam_v*tau_star)):.0f}% cheaper)')

fig,axes=plt.subplots(1,3,figsize=(15,4))

# NPV vs adoption time
axes[0].plot(tau_arr+2024, npv_arr/1e3,'b-',lw=2.5)
axes[0].axvline(tau_star+2024,ls='--',color='gold',lw=2,
                 label=f'Optimal τ*={tau_star:.1f}yr ({int(tau_star+2024)})')
axes[0].axhline(0,color='k',lw=1)
for atype,ap in adopter_types.items():   # loop: mark adopter types
    npv_v=npv(ap['tau'],C0_v,lam_v,B0_v,k_v,t0_v,r_v,T_v)
    axes[0].plot(ap['tau']+2024,npv_v/1e3,marker=ap['marker'],
                  color=ap['color'],ms=ap['ms'],zorder=5,
                  label=atype.split('(')[0].strip())
axes[0].fill_between(tau_arr+2024,npv_arr/1e3,0,
    where=npv_arr>0,alpha=0.15,color='green',label='NPV>0 zone')
axes[0].set_xlabel('Adoption year'); axes[0].set_ylabel('NPV ($k)')
axes[0].set_title('NPV vs adoption timing\n(Photonic sensing HW)')
axes[0].legend(fontsize=7); axes[0].grid(True,alpha=0.3)

# Hardware cost + benefit curves
t_full = np.linspace(0,T_v,300)
cost_traj = C0_v*np.exp(-lam_v*t_full)/1e3
benefit    = B0_v/(1+np.exp(-k_v*(t_full-t0_v)))/1e3
axes[1].plot(t_full+2024,cost_traj,'r-',lw=2,label='Purchase cost ($k)')
axes[1].plot(t_full+2024,benefit,'g-',lw=2,label='Annual benefit ($k)')
axes[1].axvline(tau_star+2024,ls='--',color='gold',lw=1.5)
# Crossover
cross_idx = np.argmin(np.abs(cost_traj-benefit))
axes[1].scatter(t_full[cross_idx]+2024,cost_traj[cross_idx],
                c='orange',s=100,zorder=5,label='Cost=Benefit')
axes[1].set_xlabel('Year'); axes[1].set_ylabel('$k')
axes[1].set_title('Hardware cost decay vs benefit growth')
axes[1].legend(fontsize=9); axes[1].grid(True,alpha=0.3)

# S-curve + adopter segments
F_full = 100/(1+np.exp(-k_v*(t_full-t0_v)))
axes[2].plot(t_full+2024,F_full,'b-',lw=2.5)
segments = [(0,2.5,'#e41a1c','Innovators'),
             (2.5,16,'#ff7f00','Early Adopt.'),
             (16,50,'#4daf4a','Early Majority'),
             (50,84,'#377eb8','Late Majority'),
             (84,100,'#984ea3','Laggards')]
for lo,hi,col,lbl in segments:     # loop: shade segments
    # find t where F crosses lo, hi
    t_lo_i = np.argmin(np.abs(F_full-lo))
    t_hi_i = np.argmin(np.abs(F_full-hi))
    axes[2].axvspan(t_full[t_lo_i]+2024,t_full[t_hi_i]+2024,alpha=0.2,color=col,label=lbl)
axes[2].set_xlabel('Year'); axes[2].set_ylabel('Market adoption (%)')
axes[2].set_title('Rogers S-curve diffusion')
axes[2].legend(fontsize=7,loc='upper left'); axes[2].grid(True,alpha=0.3)
plt.suptitle('💰 Honest Adoption Economics: When to Buy Photonic Hardware',
             fontsize=11,fontweight='bold')
plt.tight_layout(); plt.show()

---
# §2 🎮 2D Wizard Game — Projectile Physics · Robot AI · Skin Passport · Dungeon

Full 2D game simulation in NumPy: wizard player, Overwatch-style robot enemies,
projectile arcs with drag, procedural dungeon, skin/passport unlock system.

In [ ]:
# ── Skin passport system ──────────────────────────────────────────
import dataclasses, hashlib
from typing import List, Dict

@dataclasses.dataclass
class Skin:
    name: str; rarity: str; color: tuple; unlock_xp: int
    def passport_hash(self):
        data=f'{self.name}{self.rarity}{self.unlock_xp}'
        return hashlib.sha256(data.encode()).hexdigest()[:12]

SKINS = [
    Skin('Arcane Scholar',   'Common',    (0.5,0.3,0.8), 0),
    Skin('Storm Caller',     'Rare',      (0.2,0.6,1.0), 1000),
    Skin('Void Walker',      'Epic',      (0.1,0.0,0.3), 5000),
    Skin('Solar Phoenix',    'Legendary', (1.0,0.6,0.0), 15000),
    Skin('Lunar Oni',        'Legendary', (0.0,0.8,0.9), 20000),
    Skin('Photon Mage',      'Mythic',    (0.0,1.0,0.8), 50000),  # Jalali collab
    Skin('Neon Dragon',      'Mythic',    (0.8,0.0,0.5), 50000),
]

class PlayerPassport:
    def __init__(self, name, xp=0):
        self.name=name; self.xp=xp
        self.unlocked=[s for s in SKINS if s.unlock_xp<=xp]
        self.active_skin=self.unlocked[-1] if self.unlocked else SKINS[0]

    def equip(self, skin_name):
        for s in self.unlocked:
            if s.name==skin_name: self.active_skin=s; return f'Equipped: {s.name}'
        return f'{skin_name} locked (need {next((s.unlock_xp for s in SKINS if s.name==skin_name),"?"):.0f} XP)'

    def passport_card(self):
        return (f'[SKIN PASSPORT] {self.name}  XP:{self.xp:,}\n'
                f'  Active: {self.active_skin.name} ({self.active_skin.rarity})\n'
                f'  Unlocked: {len(self.unlocked)}/{len(SKINS)} skins\n'
                f'  Hash: {self.active_skin.passport_hash()}')

# Demo passports
players=[PlayerPassport('WizardKing_MLK',22000),
         PlayerPassport('PhotonMaster',55000),
         PlayerPassport('NightOwl',3000)]
for p in players: print(p.passport_card(),end='\n\n')  # loop: print passports

# ── 2D Physics: projectile with drag + gravity ─────────────────────
def spell_trajectory(pos0, vel0, drag_coeff=0.02, gravity=9.81,
                     dt=0.02, max_t=5.0):
    """2D projectile: pos0=(x,y), vel0=(vx,vy). Returns trajectory arrays."""
    pos=np.array(pos0,float); vel=np.array(vel0,float)
    xs=[pos[0]]; ys=[pos[1]]; ts=[0.0]; t=0.0
    while t<max_t and pos[1]>=0:  # loop: physics integration
        speed   = np.linalg.norm(vel)+1e-10
        drag    = -drag_coeff*speed*vel
        gravity_v=np.array([0,-gravity])
        acc     = drag+gravity_v
        vel    += acc*dt; pos+=vel*dt; t+=dt
        xs.append(pos[0]); ys.append(pos[1])
    return np.array(xs),np.array(ys),np.array(ts[:len(xs)])

# Spell types: different speeds, drags
spells = [
    ('⚡ Lightning Bolt',  (0,1), (25,15), 0.01, '#00ccff'),
    ('🔥 Fireball',        (0,1), (18,20), 0.04, '#ff4400'),
    ('🌊 Tidal Wave',       (0,1), (12,18), 0.08, '#0066ff'),
    ('💀 Death Shard',      (0,1), (30,10), 0.005,'#9900ff'),
    ('🌙 Moon Beam',        (0,1), (22,22), 0.02, '#ccaaff'),
]

# ── Robot enemy AI (finite state machine) ─────────────────────────
class RobotEnemy:
    STATES = ['patrol','chase','attack','retreat','dead']

    def __init__(self, pos, hp=100, speed=3.0, aggro_range=15,
                 attack_range=5, robot_type='tank'):
        self.pos=np.array(pos,float); self.hp=hp
        self.speed=speed; self.aggro=aggro_range
        self.atk_r=attack_range; self.state='patrol'
        self.rtype=robot_type
        self.patrol_dir=np.random.uniform(-1,1,2)
        self.patrol_dir/=np.linalg.norm(self.patrol_dir)+1e-10
        self.history=[pos[:]]

    def update(self, player_pos, dt=0.1):
        d=np.linalg.norm(player_pos-self.pos)
        # FSM transitions
        if self.hp<=0: self.state='dead'; return
        if d<self.atk_r: self.state='attack'
        elif d<self.aggro: self.state='chase'
        elif self.state not in ('patrol','dead'): self.state='patrol'

        if self.state=='patrol':     # loop: wander
            self.pos+=self.patrol_dir*self.speed*0.5*dt
            if np.random.rand()<0.05: # random direction change
                self.patrol_dir=np.random.uniform(-1,1,2)
                self.patrol_dir/=np.linalg.norm(self.patrol_dir)+1e-10
        elif self.state=='chase':
            dir_v=(player_pos-self.pos)/(d+1e-10)
            self.pos+=dir_v*self.speed*dt
        elif self.state=='attack':
            pass  # melee in range
        self.history.append(self.pos.copy())

    def take_damage(self, dmg):
        self.hp=max(0,self.hp-dmg)
        if d2:=np.linalg.norm(np.random.randn(2)); self.hp<40:
            self.state='retreat'

# Robot types
robot_roster=[
    RobotEnemy([20,8],  hp=200, speed=2.0, aggro_range=20, robot_type='tank'),
    RobotEnemy([35,15], hp=80,  speed=5.0, aggro_range=25, robot_type='scout'),
    RobotEnemy([15,25], hp=120, speed=3.0, aggro_range=18, robot_type='sniper'),
    RobotEnemy([45,5],  hp=60,  speed=6.0, aggro_range=30, robot_type='assassin'),
]

# Simulate 200 ticks of combat
player_pos=np.array([5.0,5.0])
player_path=[player_pos.copy()]
for tick in range(200):                # loop: game ticks
    # Player moves in figure-8
    theta_t=tick*0.05
    player_pos=np.array([25+20*np.sin(theta_t), 15+10*np.sin(2*theta_t)])
    player_path.append(player_pos.copy())
    for robot in robot_roster:         # loop: update robots
        robot.update(player_pos, dt=0.1)
        if robot.state=='attack' and robot.hp>0:
            robot.take_damage(np.random.randint(5,25))  # player hits back

# ── Procedural dungeon (BSP room partition) ────────────────────────
def bsp_dungeon(W=60,H=40,min_room=6,seed=42):
    rng_d=np.random.default_rng(seed)
    grid=np.zeros((H,W),dtype=int)  # 0=wall, 1=floor, 2=door, 3=chest

    def split_rect(x,y,w,h,depth=0):
        """Recursively split rectangle. Returns list of rooms."""
        if depth>5 or w<min_room*2 or h<min_room*2: return [(x,y,w,h)]
        horiz = rng_d.random()<0.5 if w>h else False
        if horiz:
            cut=rng_d.integers(min_room, h-min_room)
            return split_rect(x,y,w,cut,depth+1)+split_rect(x,y+cut,w,h-cut,depth+1)
        else:
            cut=rng_d.integers(min_room, w-min_room)
            return split_rect(x,y,cut,h,depth+1)+split_rect(x+cut,y,w-cut,h,depth+1)

    rects=split_rect(1,1,W-2,H-2)
    rooms=[]
    for (rx,ry,rw,rh) in rects:       # loop: carve rooms
        pad=1
        x1,y1=rx+pad, ry+pad
        x2,y2=min(rx+rw-pad,W-1), min(ry+rh-pad,H-1)
        if x2-x1>2 and y2-y1>2:
            grid[y1:y2, x1:x2]=1
            rooms.append((x1,y1,x2,y2))
            # Chest in 30% of rooms
            if rng_d.random()<0.3:
                cx=rng_d.integers(x1+1,x2-1); cy=rng_d.integers(y1+1,y2-1)
                grid[cy,cx]=3

    # Connect rooms with corridors
    for i in range(len(rooms)-1):     # loop: connect rooms
        ax=(rooms[i][0]+rooms[i][2])//2; ay=(rooms[i][1]+rooms[i][3])//2
        bx=(rooms[i+1][0]+rooms[i+1][2])//2; by=(rooms[i+1][1]+rooms[i+1][3])//2
        for x in range(min(ax,bx),max(ax,bx)+1): grid[ay,x]=1  # horiz
        for y in range(min(ay,by),max(ay,by)+1): grid[y,bx]=1  # vert
        # Door at junction
        grid[ay,bx]=2
    return grid, rooms

dungeon, rooms = bsp_dungeon(W=80,H=50)

# ── Visualize everything ──────────────────────────────────────────
fig=plt.figure(figsize=(16,10))
gs_game=gridspec.GridSpec(2,3,figure=fig,hspace=0.4,wspace=0.35)

# Spell trajectories
ax_spell=fig.add_subplot(gs_game[0,0])
for name,pos0,vel0,drag,color in spells:   # loop: draw spells
    xs,ys,_=spell_trajectory(pos0,vel0,drag)
    ax_spell.plot(xs,ys,color=color,lw=2,label=name.split()[1])
    ax_spell.scatter(xs[-1],ys[-1],color=color,s=50,zorder=5)
ax_spell.set_xlabel('Range (m)'); ax_spell.set_ylabel('Height (m)')
ax_spell.set_title('Spell trajectories (drag+gravity)')
ax_spell.legend(fontsize=7); ax_spell.grid(True,alpha=0.3)
ax_spell.set_ylim(bottom=0)

# Combat simulation
ax_comb=fig.add_subplot(gs_game[0,1])
pp=np.array(player_path)
ax_comb.plot(pp[:,0],pp[:,1],'k-',lw=1.5,alpha=0.5,label='Player')
ax_comb.scatter(*pp[0],'k^',s=100,zorder=6)
colors_r={'tank':'red','scout':'orange','sniper':'purple','assassin':'darkred'}
for rob in robot_roster:           # loop: plot robot paths
    rh=np.array(rob.history)
    col=colors_r[rob.rtype]
    ax_comb.plot(rh[:,0],rh[:,1],color=col,lw=1,alpha=0.6)
    ax_comb.scatter(*rh[-1],color=col,s=80,
                     marker='x' if rob.hp==0 else 'o',zorder=5,
                     label=f'{rob.rtype} HP:{rob.hp}')
ax_comb.set_title('Robot enemy AI (FSM)')
ax_comb.legend(fontsize=7); ax_comb.grid(True,alpha=0.3)

# Dungeon
ax_dun=fig.add_subplot(gs_game[0,2])
cmap_d=plt.cm.colors.ListedColormap(['#1a0a00','#c8a46e','#4488ff','#ffd700'])
ax_dun.imshow(dungeon,cmap=cmap_d,vmin=0,vmax=3,interpolation='nearest',origin='lower')
ax_dun.set_title(f'BSP dungeon ({len(rooms)} rooms)')
ax_dun.axis('off')
# Spawn wizard
if rooms: ax_dun.scatter((rooms[0][0]+rooms[0][2])//2,
                           (rooms[0][1]+rooms[0][3])//2,
                           marker='$🧙$',s=400,zorder=6)

# Skin passport visual
ax_skin=fig.add_subplot(gs_game[1,:])
n_skins=len(SKINS)
rarity_order={'Common':0,'Rare':1,'Epic':2,'Legendary':3,'Mythic':4}
xp_max=55000
for i,s in enumerate(SKINS):           # loop: draw skin cards
    xn=i/(n_skins-1); yn=rarity_order[s.rarity]/4
    rect=mpatches.FancyBboxPatch((xn-0.07,yn-0.12),0.14,0.24,
        boxstyle='round,pad=0.02',facecolor=s.color,
        edgecolor='gold' if s.rarity in ('Legendary','Mythic') else 'gray',
        linewidth=2 if s.rarity in ('Legendary','Mythic') else 1,
        transform=ax_skin.transData)
    ax_skin.add_patch(rect)
    ax_skin.text(xn,yn+0.05,s.name,ha='center',va='bottom',fontsize=7,
                  fontweight='bold',color='white',
                  path_effects=[pe.withStroke(linewidth=2,foreground='black')])
    ax_skin.text(xn,yn-0.05,f'{s.unlock_xp//1000}k XP',
                  ha='center',va='top',fontsize=7,color='white')
ax_skin.set_xlim(-0.1,1.1); ax_skin.set_ylim(-0.2,1.3)
ax_skin.set_yticks([0,0.25,0.5,0.75,1.0])
ax_skin.set_yticklabels(['Common','Rare','Epic','Legendary','Mythic'],fontsize=8)
ax_skin.set_xticks([])
ax_skin.set_title('Skin Passport — Wizard Unlock Tree')
ax_skin.grid(True,axis='y',alpha=0.3)

plt.suptitle('🎮 2D Wizard Game: Spells · Robot AI · Dungeon · Skin Passport',
             fontsize=11,fontweight='bold')
plt.tight_layout(); plt.show()

---
# §3 🤿 Photonic AUV — 3D Hull Design · Sensor Payload · Jalali Art

**Photonic AUV** = autonomous underwater vehicle with dispersion-assisted sensing.  
Payload: fiber-optic hydrophone array, TD-GS phase recovery unit, photonic IMU.  
**Buoyancy:** $F_b = \\rho_{water} V g$, neutral buoyancy at $\\rho_{AUV} = \\rho_{water}$.  
**Drag:** $F_D = \\frac{1}{2}\\rho C_D A v^2$, $C_D\\approx 0.04$ for torpedo body.

In [ ]:
# ── 3D AUV hull geometry ──────────────────────────────────────────
def auv_hull(L=2.0, D=0.25, n_theta=60, n_z=100):
    """Torpedo-shaped AUV: ogive nose + cylindrical body + cone tail."""
    theta  = np.linspace(0,2*np.pi,n_theta)
    z_full = np.linspace(-L/2,L/2,n_z)

    # Radius profile r(z)
    def r_profile(z):
        z_n = z/(L/2)   # normalized -1 to 1
        if z_n < -0.6:  # nose ogive
            t = (z_n+1)/0.4
            return D/2 * np.sqrt(1-(1-t)**2)
        elif z_n > 0.5:  # tail cone
            return D/2 * (1-z_n)/0.5
        else:            # cylinder
            return D/2

    r_vals = np.array([r_profile(z) for z in z_full])

    # Mesh
    Z,TH = np.meshgrid(z_full, theta)
    R    = np.tile(r_vals, (n_theta,1))
    X    = R*np.cos(TH); Y = R*np.sin(TH)
    return X,Y,Z,r_vals,z_full

X_hull,Y_hull,Z_hull,r_vals,z_vals = auv_hull()

# ── Sensor payload layout ─────────────────────────────────────────
sensors = [
    ('TD-GS Phase Unit',  0.0,  0.0,   0.05, 'gold',    's',  0.10),
    ('Hydrophone Array',  0.0,  0.08, -0.30, '#00ccff', '^',  0.06),
    ('Photonic IMU',      0.0, -0.08,  0.20, '#ff6600', 'D',  0.05),
    ('Fiber Spool',       0.0,  0.0,   0.40, '#aaffaa', 'o',  0.08),
    ('Camera Array',      0.0,  0.0,  -0.70, '#ffaaff', 'p',  0.07),
    ('Battery Pack',      0.0,  0.0,   0.80, '#ff4444', 'h',  0.09),
]

# ── Hydrodynamics ─────────────────────────────────────────────────
rho_water=1025; g=9.81; CD_auv=0.04
L_auv=2.0; D_auv=0.25
V_auv = np.trapz(np.pi*r_vals**2, z_vals)   # volume
A_frontal = np.pi*(D_auv/2)**2
m_auv = rho_water*V_auv  # neutral buoyancy
print(f'AUV Volume: {V_auv*1e3:.2f} L,  Mass (neutral): {m_auv:.2f} kg')
print(f'Frontal area: {A_frontal*1e4:.1f} cm²')

# Speed vs power
v_arr = np.linspace(0.1, 5.0, 200)   # m/s
F_drag = 0.5*rho_water*CD_auv*A_frontal*v_arr**2
Power  = F_drag*v_arr   # Watts

# ── Jalali photonics art ──────────────────────────────────────────
# Artistic visualization: dispersion rainbow + GS phase as mandala
N_art = 512
x_art = np.linspace(-1,1,N_art); y_art = np.linspace(-1,1,N_art)
X_art,Y_art = np.meshgrid(x_art,y_art)
R_art = np.sqrt(X_art**2+Y_art**2)+1e-10
TH_art= np.arctan2(Y_art,X_art)

# Dispersion rainbow: color = angular frequency, intensity = |J_n|
from scipy.special import jv as jv_art
art_field  = np.zeros((N_art,N_art,3))
for n_v,base_hue in enumerate([0.0,0.33,0.66]):   # loop: RGB channels
    bessel_r  = jv_art(n_v, 8*R_art)
    spiral    = np.cos(n_v*TH_art + 12*R_art)
    dispersion= np.exp(-R_art**1.5)*np.abs(bessel_r)*np.abs(spiral)
    art_field[:,:,n_v] = (dispersion-dispersion.min())/(dispersion.ptp()+1e-10)

# GS convergence mandala
mandala  = np.zeros((N_art,N_art))
for k_v in [1,2,3,5,7,11,13]:          # loop: prime harmonics
    mandala += np.cos(k_v*TH_art)*jv_art(k_v,k_v*R_art*5)*np.exp(-R_art*1.5)
mandala = (mandala-mandala.min())/(mandala.ptp()+1e-10)

# ── Plot ─────────────────────────────────────────────────────────
fig=plt.figure(figsize=(16,9))
gs_auv=gridspec.GridSpec(2,4,figure=fig,hspace=0.4,wspace=0.35)

# 3D AUV
ax3d=fig.add_subplot(gs_auv[:,0:2],projection='3d')
ax3d.plot_surface(X_hull,Y_hull,Z_hull,alpha=0.25,color='steelblue',
                   linewidth=0,antialiased=True)
# Sensor markers
for sname,sx,sy,sz,sc,smk,ssize in sensors:   # loop: plot sensors
    ax3d.scatter(sx,sy,sz,color=sc,marker=smk,s=200,zorder=10,
                  depthshade=False)
    ax3d.text(sx+0.05,sy+0.05,sz,sname.split()[0],fontsize=7,color=sc)
ax3d.set_xlabel('x'); ax3d.set_ylabel('y'); ax3d.set_zlabel('z (m)')
ax3d.set_title('Photonic AUV — 3D hull + sensor payload')
ax3d.set_box_aspect([0.5,0.5,2])

# Power vs speed
ax_pw=fig.add_subplot(gs_auv[0,2])
ax_pw.plot(v_arr,Power,'b-',lw=2); ax_pw.plot(v_arr,F_drag,'r--',lw=2,label='Drag (N)')
ax_pw.set_xlabel('Speed (m/s)'); ax_pw.set_ylabel('Power (W) / Force (N)')
ax_pw.set_title('AUV propulsion'); ax_pw.legend(fontsize=8); ax_pw.grid(True,alpha=0.3)
# Mark typical survey speeds
for v_t,lbl in [(1.5,'Survey'),(3.0,'Sprint')]:   # loop
    P_t=0.5*rho_water*CD_auv*A_frontal*v_t**2*v_t
    ax_pw.scatter(v_t,P_t,s=80,zorder=5)
    ax_pw.annotate(f'{lbl}\n{P_t:.0f}W',(v_t,P_t),textcoords='offset points',
                    xytext=(8,4),fontsize=8)

# Hull profile
ax_prof=fig.add_subplot(gs_auv[1,2])
ax_prof.fill_betweenx(z_vals*100, -r_vals*100, r_vals*100, alpha=0.5, color='steelblue')
for _,_,_,sz,sc,smk,_ in sensors:
    ax_prof.scatter(0,sz*100,color=sc,marker='*',s=100,zorder=5)
ax_prof.set_xlabel('radius (cm)'); ax_prof.set_ylabel('z (cm)')
ax_prof.set_title('Hull cross-section'); ax_prof.grid(True,alpha=0.3)

# Jalali art: dispersion rainbow
ax_art1=fig.add_subplot(gs_auv[0,3])
ax_art1.imshow(art_field,origin='lower',extent=[-1,1,-1,1])
ax_art1.set_title('Jalali Art: Dispersion Rainbow\n(Bessel × spiral × GVD)')
ax_art1.axis('off')

# Jalali art: GS mandala
ax_art2=fig.add_subplot(gs_auv[1,3])
ax_art2.imshow(mandala,cmap='twilight',origin='lower',extent=[-1,1,-1,1])
ax_art2.set_title('GS Phase Mandala\n(Prime Bessel harmonics)')
ax_art2.axis('off')

plt.suptitle('🤿 Photonic AUV + Jalali Photonics Art',fontsize=11,fontweight='bold')
plt.tight_layout(); plt.show()

---
# §4 🌙 Fly Me to the Moon — Hohmann Transfer · ΔV Budget · Lunar Trajectory

**Hohmann transfer** Earth→Moon: two-impulse maneuver minimizing ΔV.  
$$\\Delta v_1 = \\sqrt{\\frac{\\mu_{Earth}}{r_1}}\\left(\\sqrt{\\frac{2r_2}{r_1+r_2}}-1\\right),\\quad
\\Delta v_2 = \\sqrt{\\frac{\\mu_{Earth}}{r_2}}\\left(1-\\sqrt{\\frac{2r_1}{r_1+r_2}}\\right)$$

*"Fly me to the moon, let me play among the stars..."* — Frank Sinatra (1954)  
→ Apollo: $\\Delta v_{total}\\approx 3.9$ km/s (LEO→TLI), transit time ≈ 3 days.

In [ ]:
# ── SymPy: Hohmann transfer ──────────────────────────────────────
mu_s,r1_s,r2_s = sp.symbols('mu r_1 r_2', positive=True)
dv1 = sp.sqrt(mu_s/r1_s)*(sp.sqrt(2*r2_s/(r1_s+r2_s))-1)
dv2 = sp.sqrt(mu_s/r2_s)*(1-sp.sqrt(2*r1_s/(r1_s+r2_s)))
dv_total = dv1+dv2
T_transfer = sp.pi*sp.sqrt((r1_s+r2_s)**3/(8*mu_s))  # half-period
print('Hohmann transfer ΔV:')
display(Math(r'\Delta v_1 = '+sp.latex(sp.simplify(dv1))))
display(Math(r'\Delta v_2 = '+sp.latex(sp.simplify(dv2))))
display(Math(r'T_{transit} = '+sp.latex(T_transfer)+r'\quad (\text{half ellipse period})'))

# ── Numerical: Earth-Moon transfer ────────────────────────────────
mu_E   = 3.986e14   # m³/s² Earth GM
R_E    = 6.371e6    # m Earth radius
R_moon = 3.844e8    # m Moon semi-major axis
h_LEO  = 400e3      # m LEO altitude
r1_v   = R_E + h_LEO
r2_v   = R_moon

dv1_v  = np.sqrt(mu_E/r1_v)*(np.sqrt(2*r2_v/(r1_v+r2_v))-1)
dv2_v  = np.sqrt(mu_E/r2_v)*(1-np.sqrt(2*r1_v/(r1_v+r2_v)))
T_h    = np.pi*np.sqrt((r1_v+r2_v)**3/(8*mu_E))

print(f'\nEarth→Moon Hohmann Transfer:')
print(f'  r₁ (LEO):    {r1_v/1e3:.0f} km = R_Earth + {h_LEO/1e3:.0f} km')
print(f'  r₂ (Moon):   {r2_v/1e6:.3f} Mm')
print(f'  Δv₁ (TLI):   {dv1_v/1e3:.3f} km/s')
print(f'  Δv₂ (LOI):   {dv2_v/1e3:.3f} km/s')
print(f'  Δv_total:    {(dv1_v+dv2_v)/1e3:.3f} km/s')
print(f'  Transit:     {T_h/3600:.1f} hr = {T_h/86400:.2f} days')

# ── Trajectory (patched conic, 2D) ────────────────────────────────
# Elliptical transfer orbit in ECI frame
a_transfer = (r1_v+r2_v)/2
e_transfer = (r2_v-r1_v)/(r2_v+r1_v)
T_orb      = 2*np.pi*np.sqrt(a_transfer**3/mu_E)

# Parametric: r(θ) = a(1-e²)/(1+e cosθ)
theta_traj = np.linspace(0,np.pi,400)   # 0→π (half ellipse)
p_traj     = a_transfer*(1-e_transfer**2)
r_traj     = p_traj/(1+e_transfer*np.cos(theta_traj))
x_traj     = r_traj*np.cos(theta_traj)
y_traj     = r_traj*np.sin(theta_traj)

# Moon orbit (circular)
th_moon = np.linspace(0,2*np.pi,500)
x_moon_orb = R_moon*np.cos(th_moon)
y_moon_orb = R_moon*np.sin(th_moon)

# Earth orbit (LEO)
x_leo = r1_v*np.cos(th_moon)
y_leo = r1_v*np.sin(th_moon)

# ── ΔV map: contour for arbitrary r1, r2 ─────────────────────────
r1_range = np.linspace(R_E+200e3, R_E+2000e3, 80)   # LEO range
r2_range = np.linspace(1e7, 4.5e8, 80)               # target range
R1G,R2G = np.meshgrid(r1_range,r2_range)
DV1G = np.sqrt(mu_E/R1G)*(np.sqrt(2*R2G/(R1G+R2G))-1)
DV2G = np.sqrt(mu_E/R2G)*(1-np.sqrt(2*R1G/(R1G+R2G)))
DV_TOT = (DV1G+DV2G)/1e3  # km/s

fig,axes=plt.subplots(1,3,figsize=(15,5))

# Trajectory
ax_traj=axes[0]
ax_traj.plot(x_leo/1e6,y_leo/1e6,'b-',lw=0.8,alpha=0.6,label='LEO')
ax_traj.plot(x_moon_orb/1e6,y_moon_orb/1e6,'silver',lw=0.8,alpha=0.6,label="Moon orbit")
ax_traj.plot(x_traj/1e6,y_traj/1e6,'gold',lw=2.5,label='Transfer orbit')
ax_traj.scatter([0],[0],color='blue',s=200,zorder=6,label='Earth')
ax_traj.scatter([R_moon/1e6],[0],color='gray',s=100,zorder=6,label='Moon')
ax_traj.scatter([x_traj[0]/1e6],[y_traj[0]/1e6],color='green',s=80,
                  marker='^',zorder=7,label=f'TLI Δv={dv1_v/1e3:.2f}km/s')
ax_traj.scatter([x_traj[-1]/1e6],[y_traj[-1]/1e6],color='red',s=80,
                  marker='v',zorder=7,label=f'LOI Δv={dv2_v/1e3:.2f}km/s')
ax_traj.set_aspect('equal'); ax_traj.grid(True,alpha=0.2)
ax_traj.set_xlabel('x (Mm)'); ax_traj.set_ylabel('y (Mm)')
ax_traj.set_title(f'Earth→Moon Hohmann\nTransit: {T_h/86400:.2f} days')
ax_traj.legend(fontsize=7)

# ΔV contour map
cs=axes[1].contourf(r1_range/1e3-R_E/1e3,r2_range/1e6,DV_TOT,levels=20,cmap='viridis')
axes[1].scatter([h_LEO/1e3],[R_moon/1e6],c='white',s=120,
                  marker='*',zorder=5,label=f'Apollo: {(dv1_v+dv2_v)/1e3:.2f} km/s')
plt.colorbar(cs,ax=axes[1],label='ΔV_total (km/s)')
axes[1].set_xlabel('Parking orbit altitude (km)')
axes[1].set_ylabel('Target orbit radius (Mm)')
axes[1].set_title('ΔV map: Hohmann Earth→target')
axes[1].legend(fontsize=8)

# Mission timeline (Frank Sinatra style)
ax_time=axes[2]; ax_time.set_xlim(0,10); ax_time.set_ylim(-1,9)
ax_time.axis('off')
mission_events=[
    (0.0, 8, 'Launch from Earth\n🚀 v_launch=11.2 km/s', '#4CAF50'),
    (0.3, 6.5,f'TLI burn: Δv₁={dv1_v/1e3:.3f} km/s\n→ Transfer orbit insertion','#2196F3'),
    (3.0, 5, f'Translunar coast\n≈{T_h/86400:.2f} days in vacuum','#9C27B0'),
    (6.0, 3.5,f'LOI burn: Δv₂={dv2_v/1e3:.3f} km/s\n→ Lunar orbit capture','#FF5722'),
    (8.0, 2, 'Landing / science ops\n🌙 "Let me play among the stars"','#FFC107'),
    (9.5, 0.5,'Return + reentry\nv_reentry≈11 km/s','#F44336'),
]
for xi,yi,txt,col in mission_events:   # loop: timeline events
    ax_time.scatter(xi,yi,s=150,color=col,zorder=5)
    ax_time.text(xi+0.15,yi,txt,fontsize=8,va='center',color=col)
    if xi>0: ax_time.annotate('',xy=(xi,yi),xytext=(prev_x,prev_y),
        arrowprops=dict(arrowstyle='->',color='gray',lw=1.2))
    prev_x,prev_y=xi,yi
ax_time.set_title('🎵 "Fly Me to the Moon" — Mission Timeline',fontsize=10,fontweight='bold')
plt.suptitle('🌙 Orbital Mechanics: Hohmann Transfer Earth→Moon',fontsize=11,fontweight='bold')
plt.tight_layout(); plt.show()

---
# §5 🌏 SE Asia Space Programs — VNSC · PhilSA · Folsom CA Demographics

**Vietnam Space Center (VNSC):** est. 2011, first satellite LOTUSat-1 (2024).  
**Philippine Space Agency (PhilSA):** est. 2019, Diwata microsatellites.  
**Folsom, CA** has one of the largest Hmong/Vietnamese communities in the US —
tech workers at Intel, aerospace, and defense (MHAFB nearby).
**Immigration → talent pipeline → space program** feedback loop.

In [ ]:
# ── SE Asian space program timelines ─────────────────────────────
space_programs = {
    'Vietnam VNSC': {
        'founded': 2011,
        'satellites': [
            (2013,'VNREDSat-1',86,'EO optical',1),
            (2019,'MicroDragon',50,'Ocean color',1),
            (2024,'LOTUSat-1',570,'SAR C-band',1),
            (2026,'LOTUSat-2',600,'SAR X-band',0),  # planned
        ],
        'budget_musd': [2.5,3.0,4.0,6.0,8.0,12.0,15.0,18.0,22.0,28.0,35.0,40.0,48.0,55.0],
        'years': list(range(2011,2025)),
        'color': '#DA020E'
    },
    'Philippines PhilSA': {
        'founded': 2019,
        'satellites': [
            (2016,'Diwata-1',50,'Earth obs microsatellite',1),
            (2018,'Diwata-2',50,'Improved EO',1),
            (2020,'Maya-1',2,'CubeSat 3U',1),
            (2021,'Maya-2',2,'CubeSat 3U',1),
            (2023,'Maya-3/4',2,'CubeSat formation',1),
            (2027,'Agila-1',400,'Comms/EO',0),  # planned
        ],
        'budget_musd': [5.0,6.0,8.0,10.0,14.0],  # 2019-2024 (PhilSA era)
        'years': list(range(2019,2025)) + [2025],
        'color': '#0038A8'
    },
    'Indonesia LAPAN/BRIN': {
        'founded': 1963,
        'satellites': [
            (2000,'LAPAN-TUBSAT',47,'EO',1),
            (2007,'LAPAN-A1',57,'AIS/imaging',1),
            (2015,'LAPAN-A2',68,'Maritime AIS',1),
            (2016,'LAPAN-A3',115,'Multispectral',1),
        ],
        'budget_musd': [30]*14,
        'years': list(range(2010,2024)),
        'color': '#CE1126'
    },
}

# ── Folsom CA demographics (2023 ACS estimates) ────────────────────
folsom_demographics = {
    'White alone':       44.8,
    'Asian':             24.1,   # large Vietnamese/Hmong/Indian
    'Hispanic/Latino':   12.3,
    'Black/AA':           4.2,
    'Two or more races':  9.8,
    'Other':              4.8,
}
folsom_asian_detail = {
    'Vietnamese':   8.2,
    'Indian':       6.1,
    'Chinese':      4.0,
    'Filipino':     2.8,
    'Hmong':        1.5,
    'Korean':       0.8,
    'Japanese':     0.7,
}

# ── Immigration → STEM pipeline model ─────────────────────────────
# Logistic model: STEM workers F(t) = K / (1 + e^{-r(t-t0)})
def stem_pipeline(t_arr, K, r_stem, t0_stem, base):
    return base + K/(1+np.exp(-r_stem*(t_arr-t0_stem)))

years_pipe = np.linspace(1975,2035,300)  # post-Vietnam war
# Vietnamese-American STEM workers in aerospace/tech (Folsom region)
viet_stem = stem_pipeline(years_pipe, K=8500, r_stem=0.15, t0_stem=2000, base=500)
# Philippines diaspora STEM
phil_stem = stem_pipeline(years_pipe, K=6000, r_stem=0.12, t0_stem=1998, base=800)
# Space program investment feedback
diaspora_remit = (viet_stem+phil_stem)*0.001  # $k/yr remittance to space programs

# ── Customs flow model (passengers/yr, LAX-HAN, LAX-MNL) ──────────
# Exponential growth with COVID dip
years_customs = np.array([2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,
                           2020,2021,2022,2023,2024,2025,2026,2027])
# LAX-Hanoi/HCMC passengers (thousands)
lax_han = np.array([450,480,520,570,640,720,800,870,950,1050,
                     200,320,750,980,1100,1180,1240,1300])
# LAX-Manila
lax_mnl = np.array([900,950,1000,1080,1180,1300,1450,1560,1680,1800,
                     350,550,1400,1700,1850,1950,2050,2150])

# ── Visualize ─────────────────────────────────────────────────────
fig=plt.figure(figsize=(16,10))
gs_sea=gridspec.GridSpec(2,3,figure=fig,hspace=0.42,wspace=0.38)

# Space program budgets
ax_bud=fig.add_subplot(gs_sea[0,0])
for prog_name,prog in space_programs.items():   # loop: programs
    ax_bud.plot(prog['years'],prog['budget_musd'],
                 'o-',color=prog['color'],lw=2,ms=5,label=prog_name)
    # Mark satellite launches
    for yr,name,mass,typ,launched in prog['satellites']:
        if launched and 2010<=yr<=2025:
            # Find budget at that year
            if yr in prog['years']:
                idx=prog['years'].index(yr)
                bv=prog['budget_musd'][idx]
                ax_bud.scatter(yr,bv,color=prog['color'],s=100,
                                marker='*',zorder=6)
ax_bud.set_xlabel('Year'); ax_bud.set_ylabel('Budget (M USD)')
ax_bud.set_title('SE Asian space program budgets\n(★=satellite launch)')
ax_bud.legend(fontsize=8); ax_bud.grid(True,alpha=0.3)

# Folsom demographics
ax_dem=fig.add_subplot(gs_sea[0,1])
labels_d=list(folsom_demographics.keys())
sizes_d=list(folsom_demographics.values())
colors_d=['#AED6F1','#F4D03F','#82E0AA','#F1948A','#C39BD3','#FAD7A0']
wedges,texts,autotexts=ax_dem.pie(sizes_d,labels=labels_d,colors=colors_d,
    autopct='%1.1f%%',startangle=90,pctdistance=0.75)
for at in autotexts: at.set_fontsize(7)
ax_dem.set_title('Folsom CA demographics\n(Pop ~82k, 2023 ACS)')

# Asian breakdown bar
ax_asian=fig.add_subplot(gs_sea[0,2])
bars=ax_asian.barh(list(folsom_asian_detail.keys()),
                    list(folsom_asian_detail.values()),
                    color='#F4D03F',edgecolor='k',lw=0.5)
for bar,pct in zip(bars,folsom_asian_detail.values()):   # loop: labels
    ax_asian.text(bar.get_width()+0.05,bar.get_y()+bar.get_height()/2,
                   f'{pct:.1f}%',va='center',fontsize=9)
ax_asian.set_xlabel('% of total population')
ax_asian.set_title('Folsom CA Asian breakdown\n(% of city total)')
ax_asian.grid(True,axis='x',alpha=0.3)

# STEM pipeline
ax_stem=fig.add_subplot(gs_sea[1,0:2])
ax_stem.plot(years_pipe,viet_stem/1e3,'r-',lw=2.5,
              label='Vietnamese-American STEM (aerospace/tech)')
ax_stem.plot(years_pipe,phil_stem/1e3,'b-',lw=2.5,
              label='Filipino-American STEM')
ax_stem.plot(years_pipe,(viet_stem+phil_stem)/1e3,'k--',lw=1.5,
              label='Combined (×1k workers)')
# Key historical markers
for yr_m,lbl_m,ypos in [
    (1975,'Fall of Saigon\n→ refugee wave',0.5),
    (1986,'Doi Moi reform',2.0),
    (1995,'VN-US normalization',3.5),
    (2000,'Dot-com: STEM surge',5.5),
    (2020,'COVID',8.0),
    (2024,'LOTUSat-1',9.5)]:
    ax_stem.axvline(yr_m,ls=':',color='gray',lw=0.8,alpha=0.6)
    ax_stem.text(yr_m+0.5,ypos,lbl_m,fontsize=7,rotation=0,
                  color='dimgray',va='bottom')
ax_stem.set_xlabel('Year'); ax_stem.set_ylabel('STEM workers (×1000)')
ax_stem.set_title('SE Asian diaspora → US STEM pipeline\n(Logistic growth model)')
ax_stem.legend(fontsize=8); ax_stem.grid(True,alpha=0.3)

# Customs passenger flow
ax_cust=fig.add_subplot(gs_sea[1,2])
ax_cust.plot(years_customs,lax_han,'r-o',ms=4,lw=2,label='LAX–Hanoi/HCMC')
ax_cust.plot(years_customs,lax_mnl,'b-s',ms=4,lw=2,label='LAX–Manila')
ax_cust.fill_between(years_customs,lax_han,alpha=0.15,color='red')
ax_cust.fill_between(years_customs,lax_mnl,alpha=0.15,color='blue')
ax_cust.axvspan(2020,2021,alpha=0.2,color='gray',label='COVID-19')
ax_cust.set_xlabel('Year'); ax_cust.set_ylabel('Passengers (thousands/yr)')
ax_cust.set_title('LAX customs flow: Vietnam & Philippines\npassenger volumes')
ax_cust.legend(fontsize=8); ax_cust.grid(True,alpha=0.3)

plt.suptitle('🌏 SE Asia: Space Programs · Folsom CA · Diaspora STEM Pipeline',
             fontsize=11,fontweight='bold')
plt.tight_layout(); plt.show()

# Summary report
print('\n── SE ASIA SPACE + DIASPORA REPORT ────────────────────────────')
for prog_name,prog in space_programs.items():  # loop: print table
    launches=[s for s in prog['satellites'] if s[4]==1]
    print(f'{prog_name:28}: {len(launches)} satellites launched, '
          f'latest budget ${prog["budget_musd"][-1]}M USD')
print(f'\nFolsom CA Asian population: {sum(folsom_asian_detail.values()):.1f}% of city')
print(f'  Vietnamese: {folsom_asian_detail["Vietnamese"]:.1f}%  '
      f'Filipino: {folsom_asian_detail["Filipino"]:.1f}%  Hmong: {folsom_asian_detail["Hmong"]:.1f}%')
print(f'\n2023 LAX passenger flows:')
print(f'  LAX–Vietnam: ~{lax_han[-5]:,}k/yr  |  LAX–Philippines: ~{lax_mnl[-5]:,}k/yr')